# Synthetic Pavia-mean GMM: when ZCA wins vs band-standardization

6-component GMM (means = Pavia class means 1,3,4,5,7,8; weights = real pool
proportions; isotropic σ₁²I). Axis 1: **sep** = median mean-distance / σ₁ ∈
[2, 5, 10, 30, 100] (2 ≈ merged blob, 100 = far crisp modes). Axis 2:
**ρ = σ₂²** ∈ [0.01, 0.1, 1.0]. Detectors: `canon` (ZCA+floor, DSM) vs
`stdnp` (std + learned linear + noise-prediction). Anchors per (sep, seed):
**oracle** (true mixture score in the same LMP statistic — the ceiling) and
**AMF**. Target = bare-soil mean, θ=0.15, n=2048, 3000 ep, seeds 42–44.

**Preregistered prediction:** canon ≈ flat in sep (ZCA normalizes between-
mode scatter to unit variance, so its post-front geometry is separation-
invariant); stdnp rises with sep; crossover moves right with ρ. At sep=2
everything collapses onto AMF ≈ oracle (built-in sanity).

**Parallel execution:** `WORKERS` trainings run concurrently in threads on
one GPU (tiny models leave it mostly idle; ~3× throughput at WORKERS=3).
Each worker has its own seeded `torch.Generator`; model init happens under a
lock with the run's seed, so data/init/results are reproducible — but noise
REALIZATIONS differ from a sequential run (statistically equivalent).
Resume-safe. ~20–30 min at WORKERS=3 on a T4.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, json, time, threading
import numpy as np
import torch
import torch.nn as nn
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
# ----------------- knobs -----------------
SEPS   = [2, 5, 10, 30, 100]
RHOS_G = [0.01, 0.1, 1.0]
SEEDS_G = [42, 43, 44]
N_TR, N_TE, EPOCHS_G = 2048, 2000, 3000
WORKERS = 3
OUT_G = 'results/gmm_fronts'; os.makedirs(OUT_G, exist_ok=True)

In [ ]:
# ----------------- protocol -----------------
import yaml
from repro.protocols.iid import load_hsi, _pd_at_fa
from repro.core.data import plant_targets, Whitening
from repro.core.models import ScoreNet
from repro.core.detectors import dsm_additive

cfg = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
data_, gt_ = load_hsi('repro/data/pavia-u.mat')
flat_ = data_.reshape(-1, data_.shape[-1]); g_ = gt_.flatten()
MEANS = np.stack([flat_[g_ == c].mean(0) for c in (1, 3, 4, 5, 7, 8)])
WEIGHTS = np.array([.365, .116, .169, .074, .073, .203]); WEIGHTS /= WEIGHTS.sum()
SIG_T = flat_[g_ == 6].mean(0).astype(np.float32)
D = MEANS.shape[1]
MED_SEP = float(np.median([np.linalg.norm(MEANS[i] - MEANS[j])
                           for i in range(6) for j in range(i + 1, 6)]))


def sample_gmm(n, sigma1, rng):
    ks = rng.choice(6, size=n, p=WEIGHTS)
    return (MEANS[ks] + sigma1 * rng.standard_normal((n, D))).astype(np.float32)


def true_score(X, sigma1):
    X = np.asarray(X, np.float64)
    d2 = ((X[:, None, :] - MEANS[None]) ** 2).sum(-1) / (2 * sigma1 ** 2)
    d2 -= d2.min(1, keepdims=True)
    w = WEIGHTS[None] * np.exp(-d2); w /= w.sum(1, keepdims=True)
    return (w @ MEANS - X) / sigma1 ** 2


def lmp(labels, psi_tr, psi_te, s):
    zb = psi_tr.mean(0); Cm = np.cov(psi_tr, rowvar=False)
    T = -((psi_te - zb) @ s) / np.sqrt(max(float(s @ Cm @ s), 1e-12))
    return _pd_at_fa(labels, T, 0.1)


def zca_pub(tr):
    X = np.asarray(tr, np.float64); mu = X.mean(0); Xc = X - mu
    S = Xc.T @ Xc / max(len(X) - 1, 1); S = (S + S.T) / 2
    ev, V = np.linalg.eigh(S)
    inv = 1.0 / np.sqrt(np.clip(ev, max(float(ev[-1]) * 1e-5, 1e2), None))
    return Whitening(mu.astype(np.float32),
                     (V @ np.diag(inv) @ V.T).astype(np.float32))


INIT_LOCK = threading.Lock()     # global-RNG init under lock
JSON_LOCK = threading.Lock()


def build_g(variant, tr, seed):
    with INIT_LOCK:
        torch.manual_seed(seed)
        if variant == 'canon':
            net = ScoreNet(D, [128], 'relu', whitening=zca_pub(tr))
            use_np = False
        else:
            mu = tr.astype(np.float64).mean(0)
            std = tr.astype(np.float64).std(0) + 1e-9
            dw = Whitening(mu.astype(np.float32),
                           np.diag(1.0 / std).astype(np.float32))
            net = ScoreNet(D, [128], 'relu', whitening=dw)
            net.net = nn.Sequential(nn.Linear(D, D, bias=True), *list(net.net))
            use_np = True
    return net.to(DEVICE), use_np


def run_one(task):
    variant, sep, rho_g, seed = task
    key = f'{variant}_sep{sep}_r{rho_g}_s{seed}'
    sigma1 = MED_SEP / sep
    sg = float(np.sqrt(rho_g))
    rng = np.random.default_rng(1000 * seed + sep)
    tr = sample_gmm(N_TR, sigma1, rng)
    te = sample_gmm(N_TE, sigma1, rng)
    planted, labels, _ = plant_targets(te, SIG_T, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    planted = planted.astype(np.float32)
    net, use_np = build_g(variant, tr, seed)
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(97 * seed + 13 * sep + int(1000 * rho_g))
    opt = torch.optim.Adam(net.parameters(), lr=5e-4)
    X = torch.tensor(tr).to(DEVICE)
    t0 = time.time()
    for ep in range(EPOCHS_G):
        net.train()
        perm = torch.randperm(N_TR, generator=gen, device=DEVICE)
        for i in range(0, N_TR, 512):
            b = X[perm[i:i + 512]]
            w = net.whiten(b)
            eps = torch.randn(w.shape, generator=gen, device=DEVICE) * sg
            if use_np:
                loss = ((net.net(w + eps) + eps) ** 2).sum(-1).mean()
            else:
                loss = ((net.net(w + eps) - (-eps / sg ** 2)) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    if use_np:
        with torch.no_grad():
            last = list(net.net)[-1]
            last.weight /= sg ** 2; last.bias /= sg ** 2
    net.eval()
    sc = dsm_additive(planted, tr, net, SIG_T)
    return key, {'pd': _pd_at_fa(labels, sc, 0.1),
                 'sec': round(time.time() - t0)}

In [ ]:
# ----------------- anchors + parallel sweep (resume-safe) -----------------
from concurrent.futures import ThreadPoolExecutor, as_completed

gp = os.path.join(OUT_G, 'metrics.json')
gr = json.load(open(gp)) if os.path.exists(gp) else {}
for sep in SEPS:                                  # anchors (fast, sequential)
    sigma1 = MED_SEP / sep
    for seed in SEEDS_G:
        dkey = f'anchors_sep{sep}_s{seed}'
        if dkey in gr:
            continue
        rng = np.random.default_rng(1000 * seed + sep)
        tr = sample_gmm(N_TR, sigma1, rng)
        te = sample_gmm(N_TE, sigma1, rng)
        planted, labels, _ = plant_targets(te, SIG_T, cfg['amplitude'],
                                           cfg['target_fraction'],
                                           model='additive', seed=seed)
        planted = planted.astype(np.float32)
        po = lmp(labels, true_score(tr, sigma1), true_score(planted, sigma1),
                 SIG_T.astype(np.float64))
        Xc = tr - tr.mean(0)
        Si = np.linalg.inv(Xc.T.astype(np.float64) @ Xc / (N_TR - 1)
                           + 1e-6 * np.eye(D))
        Tam = (planted - tr.mean(0)) @ (Si @ SIG_T.astype(np.float64))
        gr[dkey] = {'oracle': po, 'amf': _pd_at_fa(labels, Tam, 0.1)}
        json.dump(gr, open(gp, 'w'))
        print(dkey, gr[dkey], flush=True)

tasks = [(v, sep, r, s) for sep in SEPS for r in RHOS_G
         for v in ('canon', 'stdnp') for s in SEEDS_G
         if f'{v}_sep{sep}_r{r}_s{s}' not in gr]
print(f'{len(tasks)} trainings to run, {WORKERS} workers')
t_all = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(run_one, t): t for t in tasks}
    for f in as_completed(futs):
        key, res = f.result()
        with JSON_LOCK:
            gr[key] = res
            json.dump(gr, open(gp, 'w'))
        print(key, f"Pd={res['pd']:.3f} ({res['sec']}s)", flush=True)
print(f'TOTAL {(time.time() - t_all) / 60:.1f} min')

In [ ]:
# ----------------- plots + stats -----------------
import matplotlib.pyplot as plt
from IPython.display import Image, display

gr = json.load(open(gp))
fig, axes = plt.subplots(1, len(RHOS_G), figsize=(5.2 * len(RHOS_G), 4.2),
                         sharey=True)
axes = np.atleast_1d(axes)
for ax, rho_g in zip(axes, RHOS_G):
    for variant, c in (('canon', 'tab:red'), ('stdnp', 'tab:purple')):
        m = [np.nanmean([gr[f'{variant}_sep{s_}_r{rho_g}_s{sd}']['pd']
                         for sd in SEEDS_G
                         if f'{variant}_sep{s_}_r{rho_g}_s{sd}' in gr]
                        or [np.nan]) for s_ in SEPS]
        ax.plot(SEPS, m, 'o-', color=c, lw=2, label=variant)
    orc = [np.nanmean([gr[f'anchors_sep{s_}_s{sd}']['oracle'] for sd in SEEDS_G
                       if f'anchors_sep{s_}_s{sd}' in gr] or [np.nan])
           for s_ in SEPS]
    amf = [np.nanmean([gr[f'anchors_sep{s_}_s{sd}']['amf'] for sd in SEEDS_G
                       if f'anchors_sep{s_}_s{sd}' in gr] or [np.nan])
           for s_ in SEPS]
    ax.plot(SEPS, orc, 'k--', lw=1.5, label='oracle')
    ax.plot(SEPS, amf, ':', color='tab:blue', lw=1.5, label='AMF')
    ax.set_xscale('log'); ax.set_xticks(SEPS); ax.set_xticklabels(SEPS)
    ax.minorticks_off(); ax.grid(alpha=0.3)
    ax.set_xlabel('mode separation / sigma_1'); ax.set_title(f'rho = {rho_g}')
axes[0].set_ylabel('Pd@0.1 (mean, 3 seeds)')
axes[-1].legend(fontsize=8)
fig.suptitle('Synthetic Pavia-mean GMM: ZCA vs band-std across mode overlap')
fig.tight_layout()
p = os.path.join(OUT_G, 'gmm_fronts.png')
fig.savefig(p, dpi=200); fig.savefig(p.replace('.png', '.pdf'))
plt.close(fig); display(Image(filename=p, width=1100))

print(f"{'':10s}" + ''.join(f'  sep={s_:<5}' for s_ in SEPS))
for rho_g in RHOS_G:
    for variant in ('canon', 'stdnp'):
        m = [np.nanmean([gr[f'{variant}_sep{s_}_r{rho_g}_s{sd}']['pd']
                         for sd in SEEDS_G
                         if f'{variant}_sep{s_}_r{rho_g}_s{sd}' in gr]
                        or [np.nan]) for s_ in SEPS]
        print(f'r={rho_g:<5} {variant:6s}' + ''.join(f'{v:10.3f}' for v in m))
print('oracle    ' + ''.join(
    f"{np.nanmean([gr[f'anchors_sep{s_}_s{sd}']['oracle'] for sd in SEEDS_G]):10.3f}"
    for s_ in SEPS))

In [ ]:
# ----------------- zip -----------------
!zip -qr gmm_fronts.zip results/gmm_fronts
!ls -lh gmm_fronts.zip
try:
    from google.colab import files
    files.download('gmm_fronts.zip')
except Exception as e:
    print('manual download:', e)